In [ ]:
import os, pathlib, tarfile, shutil, sys, subprocess
base='/kaggle/input/datasets/asoksci/eeg-dementia-coursach-full'
out='/kaggle/working/repo'
os.makedirs(out, exist_ok=True)
for tar in pathlib.Path(base).rglob('*.tar'):
    try:
        tf=tarfile.open(tar); tf.extractall(out); tf.close()
    except: pass
for p in ['data','configs','src','reports','figures']:
    s=pathlib.Path(base)/p
    if s.exists(): shutil.copytree(s, pathlib.Path(out)/p, dirs_exist_ok=True)
for d in list(pathlib.Path(out).rglob('src')):
    if str(d)==out+'/src': continue
    for f in d.glob('*.py'): shutil.copy(f, pathlib.Path(out)/'src'/f.name)
for d in list(pathlib.Path(out).rglob('configs')):
    if str(d)==out+'/configs': continue
    for f in d.iterdir():
        if f.is_file(): shutil.copy(f, pathlib.Path(out)/'configs'/f.name)
for d in list(pathlib.Path(out).rglob('data')):
    if str(d)==out+'/data': continue
    for f in d.iterdir():
        if f.is_file() and not (pathlib.Path(out)/'data'/f.name).exists(): shutil.copy(f, pathlib.Path(out)/'data'/f.name)
for d in list(pathlib.Path(out).rglob('reports')):
    if str(d)==out+'/reports': continue
    for f in d.iterdir():
        if f.is_file(): shutil.copy(f, pathlib.Path(out)/'reports'/f.name)
for d in list(pathlib.Path(out).rglob('figures')):
    if str(d)==out+'/figures': continue
    for f in d.iterdir():
        if f.is_file(): shutil.copy(f, pathlib.Path(out)/'figures'/f.name)
os.chdir(out)
!cat > src/run_pipeline.py << 'PYEOF'
import subprocess, sys, pathlib
steps=[["python","src/05_train_eval_real.py"],["python","src/11_stats.py"],["python","src/12_plots.py"],["python","src/13_baseline_nn.py"],["python","src/15_loso_eval.py"],["python","src/17_mlp_loso.py"],["python","src/18_compare_protocols.py"],["python","src/19_loso_plots.py"],["python","src/20_export_tables.py"]]
for cmd in steps:
    print('>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True)
print('pipeline done')
PYEOF
print(open('src/run_pipeline.py').read())
!pip install -q mne torch scikit-learn seaborn -q
!python src/run_pipeline.py
print('done')
